In [ ]:
!pip install -qU transformers datasets optimum
!pip install -qU openai wandb
!pip install -qU json-repair
!pip install -qU faker
!pip install -qU vllm
!pip install -qU unsloth bitsandbytes accelerate peft trl

In [ ]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('huggingface')
!huggingface-cli login --token {hf_token}

In [ ]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from google.colab import drive


drive.mount("/content/drive")

data_dir = "/content/drive/MyDrive"
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda"
torch_dtype = None

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

In [ ]:
raw_data_path = join(data_dir, "datasets", "news-sample.jsonl")

raw_data = []
for line in open(raw_data_path):
    if line.strip() == "":
        continue

    raw_data.append(
        json.loads(line.strip())
    )

random.Random(101).shuffle(raw_data)

print(f"Raw data: {len(raw_data)}")

In [ ]:
import os
import requests
import json
from tqdm.auto import tqdm
from os.path import join

# 🔹 OpenRouter API
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")  # set in your .env / environment
model_id = "deepseek/deepseek-v3.1-terminus"  # more reliable free model

# ✅ Ensure datasets folder exists
data_dir = os.getcwd()
os.makedirs(join(data_dir, "datasets"), exist_ok=True)

save_to = join(data_dir, "datasets", "sft.jsonl")

def try_fix_json(text: str):
    """Attempt to clean and parse JSON from text response."""
    try:
        text = text.strip()
        if text.startswith("```"):
            parts = text.split("```")
            if len(parts) >= 2:
                text = parts[1]
        text = text.replace("\n", " ").strip()
        return json.loads(text)
    except Exception:
        return None

ix = 0
for story in tqdm(raw_data):

    sample_details_extraction_messages = [
        {
            "role": "system",
            "content": "\n".join([
                "You are an NLP data parser.",
                "You will be provided with an Arabic text associated with a Pydantic schema.",
                "Generate the output in the same story language.",
                "You have to extract JSON details from text according to the Pydantic details.",
                "Extract details as mentioned in text.",
                "Do not generate any introduction or conclusion."
            ])
        },
        {
            "role": "user",
            "content": "\n".join([
                "## Story:",
                story['content'].strip(),
                "",
                "## Pydantic Details:",
                json.dumps(NewsDetails.model_json_schema(), ensure_ascii=False),
                "",
                "## Story Details:",
                "```json"
            ])
        }
    ]

    # 🔹 Send request to OpenRouter
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": model_id,
                "messages": sample_details_extraction_messages,
                "temperature": 0.2,
                "max_tokens": 512
            },
            timeout=60
        )
        result = response.json()
    except Exception as e:
        result = {"error": str(e)}

    llm_response = None
    llm_resp_dict = None

    if "choices" in result:
        llm_response = result["choices"][0]["message"]["content"]

        # Try parsing JSON
        try:
            llm_resp_dict = parse_json(llm_response)
        except NameError:
            llm_resp_dict = None

        if not llm_resp_dict:
            llm_resp_dict = try_fix_json(llm_response)

    # ✅ Save everything (success or fail)
    record = {
        "id": ix,
        "story": story['content'].strip(),
        "task": "Extract the story details into a JSON.",
        "output_scheme": json.dumps(NewsDetails.model_json_schema(), ensure_ascii=False),
        "status": "success" if llm_resp_dict else "failed",
        "response": llm_resp_dict if llm_resp_dict else llm_response if llm_response else result
    }

    with open(save_to, "a", encoding="utf8") as dest:
        dest.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

    ix += 1

    if ix % 3 == 0:
        print(f"Iteration {ix}: Saved {ix} stories ✅")

print(f"\n✅ Finished! Dataset at: {save_to}")


In [ ]:
import os
import requests
import json
from tqdm.auto import tqdm
from os.path import join

# 🔹 OpenRouter API
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")  # set in your .env / environment
model_id = "deepseek/deepseek-v3.1-terminus"  # ✅ more reliable free model

# ✅ Ensure datasets folder exists
data_dir = os.getcwd()
os.makedirs(join(data_dir, "datasets"), exist_ok=True)

save_to = join(data_dir, "datasets", "sft.jsonl")

def try_fix_json(text: str):
    """Attempt to clean and parse JSON from text response."""
    try:
        text = text.strip()
        if text.startswith("```"):
            parts = text.split("```")
            if len(parts) >= 2:
                text = parts[1]
        text = text.replace("\n", " ").strip()
        return json.loads(text)
    except Exception:
        return None

ix = 0
for story in tqdm(raw_data):

    for targeted_lang in ["English", "French"]:
        sample_translation_messages = [
            {
                "role": "system",
                "content": "\n".join([
                    "You are a professional translator.",
                    "You will be provided with an Arabic text associated with a Pydantic schema.",
                    "You must translate the text into the `Targeted Language`.",
                    "Follow the JSON Schema strictly.",
                    "Do not generate any introduction or conclusion.",
                    "Output must be **valid JSON only**."
                ])
            },
            {
                "role": "user",
                "content": "\n".join([
                    "## Story:",
                    story['content'].strip(),
                    "",
                    "## Pydantic Details:",
                    json.dumps(TranslatedStory.model_json_schema(), ensure_ascii=False),
                    "",
                    "## Targeted Language or Dialect:",
                    targeted_lang,
                    "",
                    "## Translated Story:",
                    "```json"
                ])
            }
        ]

        # 🔹 Send request to OpenRouter
        try:
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": model_id,
                    "messages": sample_translation_messages,
                    "temperature": 0.2,
                    "max_tokens": 512
                },
                timeout=60
            )
            result = response.json()
        except Exception as e:
            result = {"error": str(e)}

        llm_response = None
        llm_resp_dict = None

        if "choices" in result:
            llm_response = result["choices"][0]["message"]["content"]

            # Try parsing JSON with parse_json() if available
            try:
                llm_resp_dict = parse_json(llm_response)
            except NameError:
                llm_resp_dict = None

            # Fallback parser
            if not llm_resp_dict:
                llm_resp_dict = try_fix_json(llm_response)

        # ✅ Save everything (success or fail)
        record = {
            "id": ix,
            "story": story['content'].strip(),
            "task": f"Translate the story into {targeted_lang} and return JSON.",
            "output_scheme": json.dumps(TranslatedStory.model_json_schema(), ensure_ascii=False),
            "status": "success" if llm_resp_dict else "failed",
            "response": llm_resp_dict if llm_resp_dict else llm_response if llm_response else result
        }

        with open(save_to, "a", encoding="utf8") as dest:
            dest.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

        ix += 1

        if ix % 3 == 0:
            print(f"Iteration {ix}: Saved {ix} translations ✅")

print(f"\n✅ Finished! Dataset at: {save_to}")


In [ ]:
# Define Pydantic models for data processing
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

StoryCategory = Literal[
    "politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"
]

EntityType = Literal[
    "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"
]

class Entity(BaseModel):
    entity_value: str = Field(..., description="The actual name or value of the entity.")
    entity_type: EntityType = Field(..., description="The type of recognized entity.")

class NewsDetails(BaseModel):
    story_title: str = Field(..., min_length=5, max_length=300,
                             description="A fully informative and SEO optimized title of the story.")
    story_keywords: List[str] = Field(..., min_items=1,
                                      description="Relevant keywords associated with the story.")
    story_summary: List[str] = Field(
                                    ..., min_items=1, max_items=5,
                                    description="Summarized key points about the story (1-5 points)."
                                )
    story_category: StoryCategory = Field(..., description="Category of the news story.")
    story_entities: List[Entity] = Field(..., min_items=1, max_items=10,
                                        description="List of identified entities in the story.")

class TranslatedStory(BaseModel):
    translated_title: str = Field(..., min_length=5, max_length=300,
                                  description="Suggested translated title of the news story.")
    translated_content: str = Field(..., min_length=5,
                                    description="Translated content of the news story.")

In [ ]:
# Load and prepare the dataset for finetuning
import json
import pandas as pd
from datasets import Dataset

# Path to your saved SFT data
sft_data_path = join(data_dir, "datasets", "sft.jsonl")

# Load the data
sft_data = []
for line in open(sft_data_path, "r", encoding="utf-8"):
    if line.strip():
        sft_data.append(json.loads(line))

# Filter successful responses
successful_data = [item for item in sft_data if item["status"] == "success"]
print(f"Total examples: {len(sft_data)}, Successful examples: {len(successful_data)}")

# Prepare the data for finetuning
chat_data = []

for item in successful_data:
    # Create a chat format that Unsloth can use
    if isinstance(item["response"], dict):
        assistant_response = json.dumps(item["response"], ensure_ascii=False)
    else:
        assistant_response = str(item["response"])
    
    chat_example = {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant that follows instructions precisely."},
            {"role": "user", "content": f"{item['story']}\n\nTask: {item['task']}"},
            {"role": "assistant", "content": assistant_response}
        ]
    }
    chat_data.append(chat_example)

# Create a HuggingFace dataset
dataset_df = pd.DataFrame(chat_data)
hf_dataset = Dataset.from_pandas(dataset_df)

# Split dataset
train_test_split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print(f"Training examples: {len(train_dataset)}, Evaluation examples: {len(eval_dataset)}")

In [ ]:
# Setup Unsloth for finetuning
from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

# Use Unsloth for model loading and optimization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_id,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    token=hf_token  # Using your HF token from earlier
)

# Configure the model for finetuning (LoRA)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,             # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)

# Define output directory for your model
output_dir = join(data_dir, "finetuned-models", "qwen-arabic-assistant")

# Setup training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="wandb",    # Using wandb for tracking
    seed=42
)

# Prepare the trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="messages",
    args=training_args,
    packing=True,
    max_seq_length=2048
)

# Train the model
trainer.train()

In [ ]:
# Save the model
final_model_path = join(data_dir, "finetuned-models", "qwen-arabic-assistant-final")

# Save the trained model
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model saved to {final_model_path}")

In [ ]:
# Test the finetuned model with inference
from unsloth import FastLanguageModel
import torch

# Load the finetuned model for inference
inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name=final_model_path,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True
)

# Function to test the model
def test_model(story, task):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that follows instructions precisely."},
        {"role": "user", "content": f"{story}\n\nTask: {task}"}
    ]
    
    prompt = inference_tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = inference_tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    
    response = inference_tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_response = response.split("ASSISTANT: ")[-1].strip()
    return assistant_response

# Test with a sample from the dataset
sample = sft_data[0]
story = sample["story"]
task = sample["task"]

print("Testing finetuned model:")
print(f"Story: {story[:100]}...")
print(f"Task: {task}")
print("\nModel Response:")
response = test_model(story, task)
print(response)

In [ ]:
# Optional: Push model to Hugging Face Hub
# Uncomment and run this if you want to share your model

# model_name = "qwen-arabic-assistant"
# model.push_to_hub(f"YOUR_HF_USERNAME/{model_name}", use_auth_token=hf_token)
# inference_tokenizer.push_to_hub(f"YOUR_HF_USERNAME/{model_name}", use_auth_token=hf_token)
# print(f"Model pushed to hub: YOUR_HF_USERNAME/{model_name}")